<a href="https://colab.research.google.com/github/Peeyusj/makemore9_10week/blob/main/makemore9_10week.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

n        = 32
n_embd   = 10
n_hidden = 64
vocab_size = 27

embcat = torch.randn(n, n_embd * 3)
Yb     = torch.randint(0, vocab_size, (n,))
bngain = torch.ones(1, n_hidden)
bnbias = torch.zeros(1, n_hidden)

# NO requires_grad here — for manual backward only
W1 = torch.randn(n_embd * 3, n_hidden) * 0.1
b1 = torch.randn(n_hidden) * 0.1
W2 = torch.randn(n_hidden, vocab_size) * 0.1
b2 = torch.randn(vocab_size) * 0.1

# Forward pass — no grad tracking
hprebn  = embcat @ W1 + b1
bnmean  = hprebn.mean(0, keepdim=True)
bnvar   = hprebn.var(0, keepdim=True, unbiased=True)
bnraw   = (hprebn - bnmean) / (bnvar + 1e-5)**0.5
hpreact = bngain * bnraw + bnbias
h       = torch.tanh(hpreact)
logits  = h @ W2 + b2
loss    = F.cross_entropy(logits, Yb)
print("loss:", loss.item())


loss: 3.458176851272583


In [76]:
# Manual backward pass
n = logits.shape[0]
probs = F.softmax(logits, dim=1)
one_hot = torch.zeros_like(probs)
one_hot[range(n), Yb] = 1
dL_dlogits = (probs - one_hot) / n

dL_dW2 = h.T @ dL_dlogits
dL_db2 = dL_dlogits.sum(dim=0)
dL_dh = dL_dlogits @ W2.T
dL_dhpreact = (1 - h**2) * dL_dh

dL_dbnraw = dL_dhpreact * bngain  # gradient through bngain first

dL_dhprebn = (1/n) * (bnvar + 1e-5)**-0.5 * (
    n * dL_dbnraw
    - dL_dbnraw.sum(0)
    - bnraw * (dL_dbnraw * bnraw).sum(0)
)

dL_dW1 = embcat.T @ dL_dhprebn
dL_db1 = dL_dhprebn.sum(dim=0)
print("manual backward done")

manual backward done


In [77]:
# Fresh copy WITH grad tracking for PyTorch
W1_pt = W1.clone().requires_grad_(True)
b1_pt = b1.clone().requires_grad_(True)
W2_pt = W2.clone().requires_grad_(True)
b2_pt = b2.clone().requires_grad_(True)

hprebn_v  = embcat @ W1_pt + b1_pt
bnmean_v  = hprebn_v.mean(0, keepdim=True)
bnvar_v   = hprebn_v.var(0, keepdim=True, unbiased=True)
bnraw_v   = (hprebn_v - bnmean_v) / (bnvar_v + 1e-5)**0.5
hpreact_v = bngain * bnraw_v + bnbias
h_v       = torch.tanh(hpreact_v)
logits_v  = h_v @ W2_pt + b2_pt
loss_v    = F.cross_entropy(logits_v, Yb)
loss_v.backward()

print(torch.allclose(dL_dW1, W1_pt.grad, atol=1e-5))
print(torch.allclose(dL_db1, b1_pt.grad, atol=1e-5))
print(torch.allclose(dL_dW2, W2_pt.grad, atol=1e-5))
print(torch.allclose(dL_db2, b2_pt.grad, atol=1e-5))
print("max diff:", (dL_dW1 - W1_pt.grad).abs().max().item())

False
True
True
True
max diff: 0.0007226113229990005
